<a href="https://colab.research.google.com/github/GabrielaRguezCampos/MiamiHeatRecommendationCupon/blob/main/Discount_Tier_Merged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Load Datasets

In [ ]:
#  cell for loading dataframes from shared google drive
import pandas as pd

# mount google drive
from google.colab import drive
drive.mount('/content/drive')

# Load each into DataFrames
checkout_df = pd.read_csv("/content/drive/MyDrive/HEAT x FIU - Temp Folder/shopify_checkouts.csv", header=None)
order_df = pd.read_csv("/content/drive/MyDrive/HEAT x FIU - Temp Folder/shopify_orders.csv", header=None)


# column assignment

checkout_columns = [
    "SeasonKey", "CheckoutHeaderKey", "CheckoutID", "ReferringSiteURLKey",
    "ReferringSiteURL", "CustomerKey", "CustomerID", "CustomerFirstName",
    "CustomerLastName", "CustomerEmail", "Source", "AbandonedCheckoutURL",
    "LandingSite", "Token", "SubTotal", "TotalPrice", "Discounts", "CreatedDateTime",
    "CheckoutLineItemKey", "ProductKey", "Quantity", "Price", "ProductID",
    "ProductTitle", "ProductDescription", "ProductTags", "ProductURL", "VariantSKU",
    "VariantID", "FullVariantTitle", "VariantURL", "CheckoutContactKey",
    "CheckoutEmail", "LastUpdateDateTime", "dwh_insert_datetime",
    "dwh_update_datetime", "TaxesAmount", "Weight", "HCS_Key"
]

order_columns = [
    "SeasonKey", "OrderHeaderKey", "CheckoutHeaderKey", "OrderKey", "OrderID",
    "OrderName", "OrderNumber", "Token", "OrderStatus", "CustomerKey", "CustomerID",
    "CustomerFirstName", "CustomerLastName", "CustomerEmail", "OrderTotalQuantity",
    "OrderSubTotal", "OrderTotalPrice", "CreatedDatetime", "OrderLineItemKey",
    "OrderLineItemID", "LineItemQuantity", "LineItemActualPrice", "LineItemPrice",
    "LineItemDiscountAmount", "LineItemTotalPrice", "ProductKey", "ProductID",
    "ProductTitle", "ProductDescription", "ProductTags", "ProductURL", "VariantSKU",
    "VariantID", "FullVariantTitle", "VariantURL", "OrderContactKey", "CheckoutEmail",
    "BillingCompany", "BillingFirstName", "BillingLastName", "BillingPhone",
    "BillingAddressID", "BillingAddress1", "BillingAddress2", "BillingLatitude",
    "BillingLongitude", "BillingCity", "BillingZipCode", "BillingProvinceCode",
    "BillingProvinceName", "BillingCountryCode", "BillingCountryName",
    "ShippingCompany", "ShippingFirstName", "ShippingLastName", "ShippingPhone",
    "ShippingAddressID", "ShippingAddress1", "ShippingAddress2", "ShippingLatitude",
    "ShippingLongitude", "ShippingCity", "ShippingZipCode", "ShippingProvinceCode",
    "ShippingProvinceName", "ShippingCountryCode", "ShippingCountryName",
    "LastUpdateDateTime", "OrderToken", "OrderShippingPrice", "rownum",
    "MasterEventKey", "dwh_insert_datetime", "dwh_update_datetime",
    "OrderCurrentSubTotal", "OrderCurrentTotalPrice", "OrderCurrentTotalDiscounts",
    "OrderCurrentTotalTax", "UpdatedDateTime", "TaxesAmount", "HCS_Key",
    "LineItemTotalTax", "OrderShippingTotalTax"
]


# Assign the column names
checkout_df.columns = checkout_columns
order_df.columns = order_columns

# Check quick overview
print("Checkouts:", checkout_df.shape)
print("Orders:", order_df.shape)

Mounted at /content/drive


<ipython-input-1-2994522888>:9: DtypeWarning: Columns (6,35) have mixed types. Specify dtype option on import or set low_memory=False.
  checkout_df = pd.read_csv("/content/drive/MyDrive/HEAT x FIU - Temp Folder/shopify_checkouts.csv", header=None)
<ipython-input-1-2994522888>:10: DtypeWarning: Columns (73,78) have mixed types. Specify dtype option on import or set low_memory=False.
  order_df = pd.read_csv("/content/drive/MyDrive/HEAT x FIU - Temp Folder/shopify_orders.csv", header=None)


Checkouts: (126310, 39)
Orders: (621172, 83)


In [ ]:
order_df['dwh_update_datetime'].describe()

,dwh_update_datetime
count,225936
unique,2278
top,2024-06-07 00:33:39.670
freq,135016


In [ ]:
# Orders
order_cols_to_keep = [
    "OrderID", "CheckoutHeaderKey", "CustomerID","VariantSKU",
    "CustomerEmail", "OrderTotalPrice","OrderSubTotal",
    "LineItemDiscountAmount",
    "CreatedDatetime", "LineItemActualPrice",
    "LineItemPrice", "LineItemDiscountAmount", "LineItemTotalPrice"
]

# Checkouts
checkout_cols_to_keep = [
    "CheckoutHeaderKey", "CustomerEmail","VariantSKU",
    "TotalPrice", "Discounts","SubTotal"
]

checkout_df['CustomerEmail'] = checkout_df['CustomerEmail'].str.strip().str.lower()
order_df['CustomerEmail'] = order_df['CustomerEmail'].str.strip().str.lower()


# 1. Merge Checkouts and Orders

In [ ]:
checkout_keys = set(checkout_df['CheckoutHeaderKey'])
order_keys = set(order_df['CheckoutHeaderKey'])

missing_keys = checkout_keys - order_keys
print(f"Missing keys from checkout_df not found in order_df: {len(missing_keys)}")


Missing keys from checkout_df not found in order_df: 46680


In [ ]:
checkout_df[checkout_df['CheckoutHeaderKey'].isin(missing_keys)].head()


,SeasonKey,CheckoutHeaderKey,CheckoutID,ReferringSiteURLKey,ReferringSiteURL,CustomerKey,CustomerID,CustomerFirstName,CustomerLastName,CustomerEmail,...,FullVariantTitle,VariantURL,CheckoutContactKey,CheckoutEmail,LastUpdateDateTime,dwh_insert_datetime,dwh_update_datetime,TaxesAmount,Weight,HCS_Key
0,2017,173341,1581075136555,13,https://www.google.com/,192046,725811724331,Trevor,Blass,trevor.g.blass@gmail.com,...,Wayne Ellington Nike Miami HEAT Youth Vice Uni...,gid://shopify/ProductVariant/40200481046662,21482,trevor.g.blass@gmail.com,2024-03-14 20:12:03.290,2024-03-19 06:12:28.150,NaN,NaN,NaN,NaN
1,2017,173342,1505561870379,13,https://www.google.com/,192003,692553121835,Valentina,RIvera,valentinarivera99@yahoo.com,...,Wayne Ellington Nike Miami HEAT Vice Uniform C...,gid://shopify/ProductVariant/40200481669254,21410,valentinarivera99@yahoo.com,2024-03-14 20:12:03.290,2024-03-19 06:12:28.150,NaN,NaN,NaN,NaN
2,2017,173343,1485896613931,13,https://www.google.com/,106767,683664179243,Simran,Agarwal,simranagarwal0916@gmail.com,...,adidas Miami HEAT Three Stripe Polo - XL / WHT,gid://shopify/ProductVariant/27889120455,21401,simranagarwal0916@gmail.com,2024-03-14 20:12:03.290,2024-03-19 06:12:28.150,NaN,NaN,NaN,NaN
3,2017,173343,1485896613931,13,https://www.google.com/,106767,683664179243,Simran,Agarwal,simranagarwal0916@gmail.com,...,New ERA HEAT Liquidsilver V-Neck - S / BLK,http://www.themiamiheatstore.com/products/heat...,21401,simranagarwal0916@gmail.com,2024-03-14 20:12:03.290,2024-03-19 06:12:28.150,NaN,NaN,NaN,NaN
4,2017,173343,1485896613931,13,https://www.google.com/,106767,683664179243,Simran,Agarwal,simranagarwal0916@gmail.com,...,Miami HEAT Clima 3Stripe Polo - XL / BLK,gid://shopify/ProductVariant/27890361863,21401,simranagarwal0916@gmail.com,2024-03-14 20:12:03.290,2024-03-19 06:12:28.150,NaN,NaN,NaN,NaN


In [ ]:
mask = checkout_df['CheckoutHeaderKey'].isin(order_df['CheckoutHeaderKey'])
print(f"Matching: {mask.sum()} / {len(checkout_df)}")


Matching: 29219 / 126310


In [ ]:
print("Order DF - SeasonKey Range:")
print("Min:", order_df['SeasonKey'].min())
print("Max:", order_df['SeasonKey'].max())


Order DF - SeasonKey Range:
Min: 2017
Max: 2024


In [ ]:
print("Checkout DF - SeasonKey Range:")
print("Min:", checkout_df['SeasonKey'].min())
print("Max:", checkout_df['SeasonKey'].max())


Checkout DF - SeasonKey Range:
Min: 2017
Max: 2024


In [ ]:
order_df_subset = order_df[order_cols_to_keep]
checkout_df_subset = checkout_df[checkout_cols_to_keep]

merged_df = pd.merge(
    order_df_subset,
    checkout_df_subset,
    on='CheckoutHeaderKey',
    how='left',
    suffixes=('_order', '_checkout')
)


In [ ]:
merged_df.shape

(704252, 18)

In [ ]:
merged_df.head(10)

,OrderID,CheckoutHeaderKey,CustomerID,VariantSKU_order,CustomerEmail_order,OrderTotalPrice,OrderSubTotal,LineItemDiscountAmount,CreatedDatetime,LineItemActualPrice,LineItemPrice,LineItemDiscountAmount,LineItemTotalPrice,CustomerEmail_checkout,VariantSKU_checkout,TotalPrice,Discounts,SubTotal
0,597843968043,275567,5095538055,113804,idiotmotionz@gmail.com,65.99,56.0,0.0,2018-06-28 08:10:49.000,0.0,28.0,0.0,28.0,NaN,NaN,NaN,NaN,NaN
1,597843968043,275567,5095538055,109559,idiotmotionz@gmail.com,65.99,56.0,0.0,2018-06-28 08:10:49.000,0.0,28.0,0.0,28.0,NaN,NaN,NaN,NaN,NaN
2,570409091115,275590,4894270407,001880,dbettan@gmail.com,60.87,48.0,0.0,2018-06-07 10:18:14.000,0.0,48.0,0.0,48.0,NaN,NaN,NaN,NaN,NaN
3,590543192107,275531,740085039147,112028,adams17@consultant.com,117.53,100.5,0.0,2018-06-22 13:14:13.000,0.0,36.0,0.0,36.0,NaN,NaN,NaN,NaN,NaN
4,590543192107,275531,740085039147,<Unknown>,adams17@consultant.com,117.53,100.5,7.5,2018-06-22 13:14:13.000,0.0,15.0,7.5,15.0,NaN,NaN,NaN,NaN,NaN
5,590543192107,275531,740085039147,<Unknown>,adams17@consultant.com,117.53,100.5,7.5,2018-06-22 13:14:13.000,0.0,15.0,7.5,15.0,NaN,NaN,NaN,NaN,NaN
6,590543192107,275531,740085039147,112017,adams17@consultant.com,117.53,100.5,0.0,2018-06-22 13:14:13.000,0.0,32.0,0.0,32.0,NaN,NaN,NaN,NaN,NaN
7,590543192107,275531,740085039147,111368,adams17@consultant.com,117.53,100.5,7.5,2018-06-22 13:14:13.000,0.0,15.0,7.5,15.0,NaN,NaN,NaN,NaN,NaN
8,590543192107,275531,740085039147,000668,adams17@consultant.com,117.53,100.5,10.0,2018-06-22 13:14:13.000,0.0,20.0,10.0,20.0,NaN,NaN,NaN,NaN,NaN
9,594294800427,275576,815835217963,111931,larrydarrington2@gmail.com,65.99,56.0,0.0,2018-06-25 17:27:22.000,0.0,28.0,0.0,28.0,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Step 5: Display total number of rows and columns
print(f"Dataset shape: {merged_df.shape[0]} rows, {merged_df.shape[1]} columns")

# Step 6: Show missing values per column
missing_values = merged_df.isnull().sum()
print("\nMissing values per column:")
print(missing_values[missing_values > 0])

# Step 7: show percentage of missing values
percent_missing = (merged_df.isnull().sum() / len(merged_df)) * 100
print("\nPercentage of missing values per column:")
print(percent_missing[percent_missing > 0].round(2))

Dataset shape: 704252 rows, 26 columns

Missing values per column:
VariantSKU_order                   6
CustomerEmail_order               15
OrderCurrentSubTotal          420378
OrderCurrentTotalPrice        420378
OrderCurrentTotalDiscounts    420378
CustomerID_checkout           592364
CustomerEmail_checkout        592464
VariantSKU_checkout           592364
TotalPrice                    592364
Discounts                     592364
SubTotal                      592364
CreatedDateTime               592364
DiscountRate                      36
dtype: int64

Percentage of missing values per column:
VariantSKU_order               0.00
CustomerEmail_order            0.00
OrderCurrentSubTotal          59.69
OrderCurrentTotalPrice        59.69
OrderCurrentTotalDiscounts    59.69
CustomerID_checkout           84.11
CustomerEmail_checkout        84.13
VariantSKU_checkout           84.11
TotalPrice                    84.11
Discounts                     84.11
SubTotal                      84.11
C

# 2. Discount Tier

## Some EDA

### Check for nulls in "LineItemTotalPrice"

In [ ]:
null = merged_df['LineItemTotalPrice'].isnull().sum()
percentage_null = (null / len(merged_df['LineItemTotalPrice'])) * 100
print(percentage_null)

0.0


#### Check for nulls in "Discounts"

In [ ]:
null = merged_df['Discounts'].isnull().sum()
percentage_null = (null / len(merged_df['Discounts'])) * 100
print(percentage_null)

84.11250518280389


### Compare "LineItemTotalPrice" to "LineItemPrice"

In [ ]:
(merged_df['LineItemTotalPrice'] == merged_df['LineItemPrice']).value_counts()



,count
True,679994
False,24258


In [ ]:
# Boolean comparison
comparison = merged_df['LineItemTotalPrice'] == merged_df['LineItemPrice']

# Percentage of False
false_percentage = (~comparison).mean() * 100
print(f"Percentage of False: {false_percentage:.2f}%")


Percentage of False: 3.44%


In [ ]:
merged_df['LineItemPriceDifference'] = merged_df['LineItemTotalPrice'] - merged_df['LineItemPrice']
difference_df = merged_df[merged_df['LineItemPriceDifference'] != 0]


In [ ]:
difference_df[['CustomerEmail_order','LineItemPrice', 'LineItemTotalPrice', 'LineItemPriceDifference']].head(20)


,CustomerEmail_order,LineItemPrice,LineItemTotalPrice,LineItemPriceDifference
25,annemcdevitt23@hotmail.com,28.00,56.00,28.00
43,shubertd3@gmail.com,30.00,120.00,90.00
64,hamza5567@gmail.com,9.99,29.97,19.98
69,hamza5567@gmail.com,9.95,29.85,19.90
71,annemcdevitt23@hotmail.com,16.00,48.00,32.00
72,annemcdevitt23@hotmail.com,40.00,80.00,40.00
74,annemcdevitt23@hotmail.com,28.00,56.00,28.00
77,annemcdevitt23@hotmail.com,19.99,39.98,19.99
80,annemcdevitt23@hotmail.com,45.00,90.00,45.00
93,smith071376@hotmail.com,30.00,90.00,60.00


### Discount Tier

In [ ]:
# Step 1: Calculate discount rate
merged_df['DiscountRate'] = (
    merged_df['OrderSubTotal'] / merged_df['OrderSubTotal']
).round(2)

# Step 2: Categorize into tiers
def get_discount_tier(rate):
    if pd.isna(rate):
        return 'Unknown'
    elif rate >= 0.8:
        return 'High'
    elif rate >= 0.3:
        return 'Medium'
    elif rate > 0:
        return 'Low'
    else:
        return 'No Discount'

merged_df['DiscountTier'] = merged_df['DiscountRate'].apply(get_discount_tier)


In [ ]:
discount_summary = merged_df['DiscountTier'].value_counts().reset_index()
discount_summary.columns = ['DiscountTier', 'Count']
discount_summary['Percentage'] = (discount_summary['Count'] / discount_summary['Count'].sum() * 100).round(2)
display(discount_summary)


,DiscountTier,Count,Percentage
0,High,704216,99.99
1,Unknown,36,0.01


In [ ]:
merged_df[['OrderSubTotal', 'OrderTotalPrice']].describe()
merged_df[['OrderSubTotal', 'OrderTotalPrice']].isna().sum()


,0
OrderSubTotal,0
OrderTotalPrice,0


In [ ]:
merged_df[['OrderSubTotal', 'OrderTotalPrice']].head(10)
merged_df[['OrderSubTotal', 'OrderTotalPrice']].value_counts().head(10)


,,count
OrderSubTotal,OrderTotalPrice,
125.0,134.99,10966
120.0,129.99,5829
35.0,44.99,3596
125.0,125.00,3010
75.0,84.99,2893
140.0,149.80,2608
35.0,48.14,2576
125.0,144.44,2450
32.0,41.99,2410
